# PV Energy Forecasting via Machine Learning

## Introduction

### Dataset Description

The dataset used in this project combines power production data from Sunnyside Solar Farm with weather data obtained from the Visual Crossing Weather API. The solar dataset contains timestamps and measured energy output from January 1, 2025 to February 9, 2026. Each entry represents a 15-minute power reading from the grid where exported solar power appears as negative values. Since the solar data is in 15-minute increments, the values will be summed into hourly energy totals to align with the weather dataset.

The weather dataset includes up to 42 environmental features. This includes solar radiation, temperature, cloud coverage, etc. All of which influence photovoltaic (PV) energy generation. Additionally, Visual Crossing provides weather forecasts up to 15 days ahead, allowing the model to potentially predict future power output based on forecasted conditions.

LPEA also indicated that some days contain inaccurate measurements or periods when the system was partially offline. Plotting the data will help identify these irregularities so that problematic periods can be evaluated and potentially excluded from the analysis.

### Problem Statement

>To what extent can an ML model predict PV energy output at Sunnyside Solar Farm whilst trained exclusively on publicly available forecast data?

### Project Goal

>To train an ML model that evaluates, ranks importance, and quantifies impacts of forecast variables on PV energy production.

## Data Loading & Exploration

In [ ]:
# all libraries used
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from Models.ModelMaker import ModelMaker
from util.plots importplot_forecasts

In [ ]:
# Shows the full 24 hours in table
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Load CSV file data 
weather_df = pd.read_csv('data/DRO_2025-01-01_2026-02-09.csv')

# Cleaning columns by removing spaces
weather_df.columns = weather_df.columns.str.strip()

# Identfy the column that contains the time information
possible_time_cols = ["datetime", "date", "time", "timestamp"]

for col in possible_time_cols:
    if col in weather_df.columns:
        time_col = col
        break
# Convert the time column to datatime format
weather_df[time_col] = pd.to_datetime(weather_df[time_col], errors="coerce")

# Removes bad timestamps and uses time column as the index
weather_df = (weather_df.dropna(subset=[time_col]).set_index(time_col).sort_index())

# Load forecast file and cleans data like weather
df_forecast = pd.read_json('data/forecast_example.json')
df_forecast.columns = df_forecast.columns.str.strip().str.lower()

# Compare the first 24 hours of each weather variable that exists in both datasets and creates table with HTML
for col in weather_df.columns:
    if col not in df_forecast.columns:
        continue

    hist = pd.to_numeric(weather_df[col].head(24), errors="coerce").reset_index(drop=True)
    forecast = pd.to_numeric(df_forecast[col].head(24), errors="coerce").reset_index(drop=True)
    time_labels = weather_df.index[:24].strftime("%m-%d %H:%M")
    comparison = pd.DataFrame([hist.round(2).values, forecast.round(2).values],index=["Historical", "Forecast"],columns=time_labels)
    
    print(f"\n{col}")

    html_table = f"""
    <div style="overflow-x:auto; width:100%; border:1px solid #ccc; padding:5px;">
        {comparison.to_html()}
    </div>
    """
    display(HTML(html_table))

After examination of features, cleaning the dataset is strictly necessary. Features such as:

- "source" and "stations" do not provide meaningful information and are thus dropped.
- "sealevelpressure" is removed because it does not relate to the forecast dataset.
- "preciptype", which contains categorical or missing values, so it must either be encoded numerically or excluded from the dataset.

### Daily & Weekly Power Plots

In [ ]:
# "power" readings from Sunnyside Solar Farm
p_data = pd.read_csv("data/SunnysideTotalPower2025-01-01_2026-02-12.csv")

# organize data
p_data["timestamp"] = pd.to_datetime(p_data["timestamp"])
p_data = p_data.set_index("timestamp")
p_data["power"] = -p_data["power"]   # since data is measured as power reduced from grid

# resample data
hourly = p_data.resample('h').mean()   # convert from W to kWh
daily = p_data.resample('D').sum()
weekly = daily.resample('W').mean()

### Plotting ###
# Daily Plot
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(daily.index, daily["power"] / 1000, linewidth=1.5)

ax.set_xlim(daily.index.min(), daily.index.max())
ax.set_title("Daily Power (kW)", fontsize=16)
ax.set_ylabel("kW", fontsize=14)
ax.set_xlabel("Date", fontsize=14)
ax.tick_params(axis='both', labelsize=12)

ax.grid(True, alpha=0.3)
plt.tight_layout()

# Weekly Plot
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(weekly.index, weekly["power"] / 1000, linewidth=2)

ax.set_xlim(weekly.index.min(), weekly.index.max())
ax.set_title("Weekly Average Power (kW)", fontsize=16)
ax.set_ylabel("kW", fontsize=14)
ax.set_xlabel("Date", fontsize=14)
ax.tick_params(axis='both', labelsize=12)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The daily power plot revealed short term variability from weather, snow cover, or system issues. The weekly plot smooths out the fluctuations from the daily plot, and visualizes larger trends depending on the weather.

In [ ]:
# Select a few important numeric features
features = ["solarradiation", "solarenergy", "cloudcover", "temp", "humidity"]

df_features = weather_df[features].dropna()

# Create scatter plots comparing features
fig, axes = plt.subplots(2, 2, figsize=(12,10))

axes[0,0].scatter(df_features["solarradiation"], df_features["solarenergy"], alpha=0.4)
axes[0,0].set_title("Solar Radiation vs Solar Energy")
axes[0,0].set_xlabel("Solar Radiation")
axes[0,0].set_ylabel("Solar Energy")

axes[0,1].scatter(df_features["cloudcover"], df_features["solarradiation"], alpha=0.4)
axes[0,1].set_title("Cloud Cover vs Solar Radiation")
axes[0,1].set_xlabel("Cloud Cover")
axes[0,1].set_ylabel("Solar Radiation")

axes[1,0].scatter(df_features["temp"], df_features["solarenergy"], alpha=0.4)
axes[1,0].set_title("Temperature vs Solar Energy")
axes[1,0].set_xlabel("Temperature")
axes[1,0].set_ylabel("Solar Energy")

axes[1,1].scatter(df_features["humidity"], df_features["cloudcover"], alpha=0.4)
axes[1,1].set_title("Humidity vs Cloud Cover")
axes[1,1].set_xlabel("Humidity")
axes[1,1].set_ylabel("Cloud Cover")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The above scatter plots show several important relationships between weather variables and solar energy production:
- Solar radiation had a strong positive linear correlation with energy.
- Cloud cover shows a negative relationship with radiation as higher cloud coverage blocks the sunlight and reduces incoming radiation.
- Temperature also shows a positive trend with energy, likely because warmer conditions are linked to clear skies and stronger sunlight.
- Humidity tends to increase with cloud cover, reflecting atmospheric conditions that form cloud formation.

In [ ]:
# Drop features that are not useful for correlation analysis.
exclude_cols = ['name', 'preciptype', 'winddir', 'source', 'stations']
df_filtered = weather_df.drop(columns=[col for col in exclude_cols if col in weather_df.columns])

# Keep only numeric columns so correlation can be calculated.
df_numeric = df_filtered.select_dtypes(include='number')

# Calculate the correlation matrix.
corr = df_numeric.corr()

# Create the plot.
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)

# Add colorbar to show correlation strength.
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Correlation", rotation=270, labelpad=15)

# Set tick marks and labels.
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

# Add correlation values inside each square.
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', fontsize=8, color='black')

# Add title.
ax.set_title("Weather Data Correlation Matrix", fontsize=16, pad=20)

plt.tight_layout()
plt.show()

Based on the correlation matrix and understanding of forecast data, the _UV index_ feature must be avoided since it is **highly** correlated to solar radiation and solar energy. (but it forecasting UV index - not sure what is meant here) This makes forecasting more difficult for our models.

The _feels-like_ feature can also be ignored since it is correlated to temperature.

## Data Preparation 

Some preprocessing was required to ensure the dataset did not contain any missing values or invalid points that might affect our models. Since the original dataset contained 15-minute increments power measurements, it was resampled into hourly averages but in order to match with the Visual Crossing data.

For feature engineering, additional variables were created to represent the physical behavior of the solar panels. PV cell temperature estimates were calculated using ambient temperatures and solar radiation, which approximated the panels operating temperature.

Since panel efficiency decreases as temp increases, it captures the temperature effects of the power output.
$$
T_{cell} = (T_{air}-32)\frac{5}{9} + \left(\frac{41-20}{800}\right)G
$$
Optimal performance was found at 41 degrees, with 20 degrees acting as a baseline of ambient conditions. The difference is then divided by 800 (based on the magnitude of solar radiation).

Heat loss was another feature added. It also represents the thermal difference between the panels and the surrounding air, which can influence the panel efficiency. To approximate this interaction, heat loss was modeled as the product of air temperature and solar radiation.
$$
HeatLoss = T_{air} \times SolarRadiation
$$
This feature helpd represent conditions where high temperature and strong sunlight together increase heat stress on the panels and reduce efficiency.

The dataset was then split into training and testing sets using 80/20 split, allowing the model to be trained on the majority of the data while having unseen data for performance evaluation. 


# Models

All models were trained using a custom grid search that evaluates the model's performance based on various scores. Most models used a 5-fold cross-validation (CV), except Ridge﷿﷿﷿which used the more efficient built-in CV﷿﷿﷿, and LSTM﷿﷿﷿due to compute limitations and given project time constraints. The models below are only snapshots of the most outstanding or unique models and their parameters based on previous results and exploration.

In [ ]:
from Models import ModelMaker


# create a model maker to demonstrate all the models cleanly
mm = ModelMaker()

# Note: if you are running the cells below on CPU (see `Available Processors` below), it may take a significant amount of time!

### Features

The code below contains the list of features that will be demonstrated in the models. Again, nearly all features were tested in the exploration of each model, but these features are the ones that stood out.

In [ ]:
features = [
    ["temp", "cloudcover", "dtemp", "dsolarradiation", "windspeed", "hcos", "hsin", "celltemp"],
    ["cloudcover", "solarradiation", "dtemp", "dsolarradiation", "windspeed", "hcos"],
    ["temp", "cloudcover", "solarradiation", "dtemp", "dsolarradiation", "solarenergy", "hcos", "hsin", "celltemp"],
    ["cloudcover", "dcloudcover", "hsin"],
    ["temp", "cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "windspeed", "hcos", "hsin", "celltemp"],
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "celltemp"],
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "celltemp", "temp"],
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "heatloss", "temp"],
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "heatloss"],
    ["temp", "solarradiation", "dcloudcover", "dsolarradiation", "hcos"],
    ["cloudcover", "dtemp", "dsolarradiation", "windspeed", "hcos", "hsin"],
    ["temp", "solarradiation", "sunelevation", "cloudcover", "sunazimuth", "solarenergy"],
]

# commonly preferred by neural nets
features_small = [
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "celltemp"],
    ["cloudcover", "solarradiation", "dcloudcover", "dtemp", "dsolarradiation", "solarenergy", "hcos", "celltemp", "temp"],
]

rs = 42   # random state

## Ridge Regression

20,000 feature combinations were ran for RidgeRegression model. The list ridge_features represents the preferred feature metrics for the ridge regression model after feature combinations were tested.

In [ ]:
ridge_params = {
    "tts": [0.2],
    "alphas": [np.logspace(-3, 3, 100)],
}

mm.train_and_eval("RidgeRegression", features, ridge_params, rs)
mm.save_best(0)

## MLP Regression

The MLP Regression model proved good at predicting ramp up and ramp down on the data, but similar to the other Neural Network below, it seems to over-predict mid-day power values.

In [ ]:
mlp_params = {
    "tts": [0.2],
    "activation": ["relu"],
    "learning_rate_init": [0.001, 0.005],
    "hidden_layer_sizes": [(64, 32), (256, 128, 64, 32)],
    "alpha": [0.001, 0.01, 0.1],
    "max_iter": [100],
    "early_stopping": [True],
    "batch_size": [64],
}

mm.train_and_eval("MLPRegression", features_small, mlp_params, rs)
mm.save_best(4)

## Random Forest

The Random Forest (RF) regression model showed strong performance for predicting solar power production using weather and solar features. The best-performing RF models used fewer input features than several other models, suggesting that a smaller set of key variables is sufficient for accurate predictions. The model achieved an $R^2$ of approximately 0.92–0.93 with relatively low error values of RMSE= 172–174 and MAE= 81–82, indicating good predictive accuracy and generalization. The predicted daily power curve closely follows the actual values and performs particularly well during the ramp-up period as solar production increases in the morning. Overall, the RF model effectively captures the nonlinear relationship between weather conditions and PV power output.

In [ ]:
rf_params = {
    "tts": [0.2],
    "n_estimators": [100, 300, 600],
    "max_depth": [15, 20],
    "min_samples_split": [2],
    "min_samples_leaf": [2, 4],
    "max_features": ["sqrt", 0.5],
    "bootstrap": [True]
}
rf_features = features_small.copy()
rf_features.append(features[-1])

mm.train_and_eval("RandomForestRegression", rf_features, rf_params, rs)
mm.save_best(0)

## Gradient Boosting

Two new parameters were added -> "validation_fraction" and "n_iter_no_change", which both govern early stopping. Saving 10% of the training data for validation proved optimal for the size of the dataset, and stopping after 20 to 40 rounds if no significant improvements showed makes sure the model has a reasonable runtime.

Testing the number of estimators up to 2000 showed that the range from 25 to 100 (generally gearing towards 100) would give the best R² value. The learning rate would work in tandem as values of 0.05 and 0.1 showed good results when paired with 100 estimators. Decreasing the learning rate and increasing the estimator count indicated diminished improvements with scores identical to the model parameters below.

A high minimum number for a tree split makes deeper learning by the boosting model possible as the maximum depth of each tree could only support up to a value of 5. Along with a high min_samples_split, the minimum samples for a leaf node came to 20 to 40 samples, and meant that each tree would be shallow in order to capture holistic trends in power production without noise and overfitting.

A subsample size of 80% prevents overfitting and assures high tree diversity (trees training on different parts of the data). Additional benefits also included noise handling as any outliers would affect a smaller number of trees.

In [ ]:
gb_params = {
              "tts": [0.2],
              "n_estimators": [25, 50, 75, 100],
              "learning_rate": [0.05, 0.1],
              "max_depth": [3, 5],
              "min_samples_split": [100, 200],
              "min_samples_leaf": [20, 40],
              "subsample": [0.8],
              "validation_fraction": [0.1],
              "n_iter_no_change": [10, 20]}

gb_features = features[6:8]

mm.train_and_eval("GradientBoostingRegression", gb_features, gb_params, rs)
mm.save_best(0)

## LSTM Regression

Long Short-Term Memory (LSTM) Regression is a type of recurrent neural network that has the unique capability to predict value sequentially. Since weather and solar energy production are time series data, the model is well-suited for the data. The model performed exceptionally well on the ramp up and ramp down phases of the daily energy curve, but oddly, the model did struggle to capture the peak of the day accurately.

While testing, some models were quite confident in their impossibly high mid-day power values. This is the exact reason that led to the creation of the cell-temp parameter. Models with higher _lookback_ (beyond 48hrs) also contributed to impossible mid-day power values. Also, since LSTM Regression depends on sequential data, there is a good chance that some of the model's testing data was a sequence of difficult days to predict, or in other words, the testing data was clumped together which made interpreting model performance difficult. A **much** larger dataset could elimiate this problem.

In [ ]:
# WARNING TO USER:
#                   THIS CELL CAN TAKE A LONG TIME TO RUN ON CPU!

lstm_params = {
    "tts": [0.2],
    "lookback": [48],
    "epochs": [500],
    "batch_size": [32],
    "lstm_units_1": [128, 64],
    "lstm_units_2": [64, 32],
    "lstm_units_3": [32, 16],
    "dense_units": [16],
    "dropout_rate": [0.3],
    "validation_split": [0.1]
}

mm.train_and_eval("LSTMRegression", [features_small[1]], lstm_params, rs)

# Even though the R2 is lower, the model with the best MAE, clearly fits the daily energy curve better.
mm.save_best(3)

## Model Comparsion

### Forcasting Example

In [ ]:
### Best Models ###
best_models = [
    mm.get_best_model("RidgeRegression"),
    mm.get_best_model("MLPRegression"),
    mm.get_best_model("RandomForestRegression"),
    mm.get_best_model("GradientBoostingRegression"),
    mm.get_best_model("LSTMRegression"),
]
# corresponding features
bm_features = [m.get_features() for m in best_models]

In [ ]:
import json
import pandas as pd
from Data.ModelData.ModelData import ModelData
from util.plots importplot_forecasts, clamp_predictions

NUM_DAYS_FORCAST = 1

with open("data/forecast_example.json", 'r') as f:
    forcast_data = json.load(f)

df = pd.read_csv("data/147_2026-02-10-2026-02-10_Sunnyside Export for FLC.csv", skiprows=[0, 1, 2, 3, 5])

# Convert Timestamp  datetime and set as index
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)

hourly_kwh = df['Production meter active power'].resample('h').mean()
actual_energy = hourly_kwh.values[:24].tolist()

# create data frame from API data
forcast_data = pd.DataFrame(forcast_data)
forcast_data["datetime"] = pd.to_datetime(forcast_data["date"] + " " + forcast_data["time"])

model_data = ModelData(forcast_data)

prediction_data = [model_data.features.copy(True)[f] for f in bm_features]
predictions = [m.predict(p_data)[:24] for p_data, m in zip(prediction_data, best_models)]

### slice data into days ###
# weather
model_data.features = model_data.features[:24]
features_days = [model_data.features[i:i+24] for i in range(0, len(model_data.features), 24)]

# predictions
days = []
for p in predictions:
    days.append([p[i:i+24] for i in range(0, len(p), 24)])

# clamp predictions
clamped = [clamp_predictions(d[0], features_days[0].index, model_data.weather["sunelevation"]) for d in days]

# clamp the actual energy (brought over from your old code)
clamped_actual = clamp_predictions(actual_energy, features_days[0].index, model_data.weather["sunelevation"])

# display results
plot_forecasts({
    "Ridge Regression": clamped[0],
    "MLP Regression": clamped[1],
    "Random Forest": clamped[2],
    "Gradient Boosting": clamped[3],
    "Actual Production": clamped_actual
}, date_str="2026-02-10")

In [ ]:
import pandas as pd

model_comp = {
    "Metric": ["R²", "CV R²", "RMSE", "RMSE Clamped", "MAE", "CI"],
    "Ridge": [0.843, "-", 248.641, 225.334, 184.941, "0.830, 0.854"],
    "MLP Regression": [0.912, 0.910, 185.720, 185.537, 95.373, "0.900, 0.924"],
    "Random Forest": [0.917, 0.925, 181.009, 181.165, 89.919, "0.905, 0.927"],
    "Gradient Boosting": [0.913, 0.918, 184.841, 184.981, 93.530, "0.900, 0.925"],
    "LSTM Regression": [0.935, "-", 137.920, 137.488, 75.055, "0.925, 0.944"]
}

df = pd.DataFrame(model_comp)
display(df)

The models show a clear difference in performance. Ridge regression performed the worst with the lowest $R^2$ of 0.843 and had the highest error value, indicating that a simple linear model does not capture the dataset's relationship. The ML models, MLP, Random Forest, and Gradient Boosting all improved performances, all having a $R^2$ value above 0.91 and lower RMSE and MAE. Notice that the Random Forest had the hight cross-validated performance with 0.925. The LSTM model performed the best overall, with the highest $R^2$ of 0.935 and the lowest RMSE of 137.92 and MAE of 75.055, indicating the most accurate predictions. 

## Feature Importance & Interpretation

Across all models, several features consistently played a major role in predicting PV power output. Solar radiation and solar energy were among the most influential variables because they directly represent the amount of incoming sunlight available to the panels, which drives power production. Sun elevation and sun azimuth were also important since they describe the position of the sun throughout the day, affecting both the intensity and angle of sunlight reaching the panels. Cloud cover influenced predictions by reducing the available solar radiation, while temperature had a smaller effect due to its impact on panel efficiency. Although the importance varied slightly depending on the model, the results consistently showed that solar irradiance and sun position were the primary drivers of PV power generation.

## Final Recommendations

### Best model

The LSTM regression model produced the best overall performance with the highest $R^2$ value and the lowest RMSE and MAE among all the testing models. This suggests the LSTM was better able to capture the patterns in the solar data compared to the other models. 

### Limitations and uncertainties

A limitation of the dataset is that random variation could still occur in the training and testing splits, which can affect model performance. In addition, the dataset does not capture real-world factors that influence solar power production, such as dust buildup, shading, maintenance, or equipment degradation. As a result, the model may not fully represent all sources of variability in real solar systems.

### Practical deployment considerations

While deploying these models in real-world solar forecasting systems, several practical factors must be considered. First, the models rely heavily on accurate weather forecasts, meaning prediction accuracy depends on the quality of incoming weather data such as solar radiation, cloud cover, and temperature. Second, seasonal variations should be considered, as solar production patterns change throughout the year and models may require periodic retraining. Computational requirements also vary between models, simple models like Ridge regression are faster and easier to develop, while neural networks and LSTM models require more processing power and longer training times. Additionally, real-world PV systems are affected by factors not included in the dataset, such as dust buildup, shading, maintenance schedules, and equipment issues, which may introduce additional uncertainty into predictions. Finally, integrating these models into operational energy systems would require automated data pipelines for weather inputs and solar production monitoring.

### Future work

- Analyze optimal PV conditions to better understand performance and system efficiency.
- Apply models to support local communities by improving energy planning, optimizing battery storge usage, and grid reliability.
- Optimize models through tuning to improve forecasting accuracy.

## Conclusion 

This project evaluated several machine learning models for predicting solar power production using weather data. While linear regression performed poorly, neural networks and other models showed improved accuracy. The LSTM model achieved the best predictive performance, demonstrating the advantage of capturing relationships in solar generation data. These results show that machine learning can be an effective tool for improving short-term solar power forecasting.